# Confusion Matrix Plots: DeepChronos Style

In [64]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pathlib import Path
import json

## Configuration

### DARPA 2000

In [65]:
# dataset = "darpa2000"
# scenario = "s1_inside"

In [66]:
# # DPL config
# logic_files = [
#     "darpa", 
#     "darpa_logic_baseline",
# ]

### AIT-LDSv2

In [67]:
dataset = "aitv2"
# scenario = "fox"
scenario = "santos"

In [68]:
logic_files = [
    "ait", 
    "ait_logic_baseline",
]

### Baselines

In [69]:
# Baseline config
baseline_models = [
    "multiclass",
    # "ensemble",
]

classes = ["benign", "phase1", "phase2", "phase3", "phase4", "phase5"]
classes = ["benign", "phase1", "phase2", "phase3", "phase4"]

## Load Data

In [70]:
experiments = {}

for logic_file in logic_files:
    # --- Load metrics ---
    metrics_dir = Path(f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/metrics")
    file_paths = list(metrics_dir.iterdir())
        
    for file_path in file_paths:
        experiment_name = str(file_path.stem)
        print(f"Processing {experiment_name}...")
        data = np.load(file_path, allow_pickle=True)
        experiments[experiment_name] = {
            "logic_file": logic_file,
            "confusion_matrix": data["confusion_matrix"],
            "classes": data["classes"].tolist(),
            "metrics": data["metrics"].item(),
        }

Processing ait_scratch_behavioral_w10_1000b1000a_20260426_135204...
Processing ait_scratch_reduced_w10_1000b1000a_20260426_142318...
Processing ait_scratch_reduced_w10_10000b10000a_20260426_140247...
Processing ait_logic_baseline_scratch_behavioral_w10_50b50a_20260428_184713...
Processing ait_logic_baseline_scratch_reduced_w10_100b100a_20260426_143747...
Processing ait_logic_baseline_scratch_behavioral_w10_50b50a_20260428_184524...


## Confusion Matrix Plotting

In [71]:
def compute_masks(cm, classes):
    n = len(classes)
    benign_idx = classes.index("benign")

    diag_mask = np.zeros_like(cm, dtype=bool)
    fp_mask = np.zeros_like(cm, dtype=bool)
    fn_mask = np.zeros_like(cm, dtype=bool)
    off_diag_mask = np.zeros_like(cm, dtype=bool)

    for i in range(n):
        for j in range(n):
            # correct prediction
            if i == j:
                diag_mask[i, j] = True
            # false alarm
            elif j == benign_idx and i != benign_idx:
                fp_mask[i, j] = True   
            # missed attack
            elif i == benign_idx and j != benign_idx:
                fn_mask[i, j] = True   
            # off diagonal but not involving benign
            else:
                off_diag_mask[i, j] = True
            
    return diag_mask, fp_mask, fn_mask, off_diag_mask

In [ ]:
def plot_confusion_matrix(
    cm,
    classes,
    out_path,
):
    """
    IDS-style confusion matrix visualization.

    Assumes:
        cm[predicted, actual]
    """

    cm = np.asarray(cm, dtype=float)

    # Normalize by column to get fractions
    col_sums = cm.sum(axis=0, keepdims=True)
    col_sums[col_sums == 0] = 1  # avoid division by zero
    cm_normalized = cm / col_sums  # values are 0–1 fractions

    classes = list(classes)

    diag_mask, fp_mask, fn_mask, off_diag_mask = compute_masks(cm, classes)
    masks = {"TP": diag_mask, "FP": fp_mask, "FN": fn_mask, "OFF-DIAG": off_diag_mask}
    cm_colors = {"TP": "Greens", "FP": "Oranges", "FN": "Oranges", "OFF-DIAG": "Reds"}

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(cm_normalized, cmap="Greys", norm=LogNorm(), interpolation="none")

    for label, mask in masks.items():
        ax.imshow(
            np.ma.masked_where(~mask, cm_normalized),
            cmap=cm_colors[label],
            norm=LogNorm(),
            interpolation="none",
            alpha=0.85,
        )
    
    ax.set_aspect("equal")

    ax.set(
        xticks=np.arange(len(classes)),
        yticks=np.arange(len(classes)),
        xticklabels=classes,
        yticklabels=classes,
        ylabel="Predicted label",
        xlabel="True label",
    )

        

    ax.set_xticks(np.arange(len(classes)+1)-.5, minor=True)
    ax.set_yticks(np.arange(len(classes)+1)-.5, minor=True)
    ax.grid(which="minor", color="lightgray", linestyle='-', linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    # Annotate cells
    thresh = cm_normalized.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):

            frac = cm_normalized[i, j]
            count = int(cm[i, j])

            text_color = "white" if frac > thresh else "black"

            ax.text(
                j,
                i,
                f"{count}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=9,
            )

    fig.tight_layout()
    
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

    print("Saved confusion matrix plot to:", out_path)

In [73]:
plots_dir = Path(f"../reports/{dataset}/{scenario}")

In [74]:
for experiment_name, data in experiments.items():
    logic_file = data["logic_file"]
    mat = data["confusion_matrix"]
    classes = data["classes"]

    cm_dir = plots_dir / "deepproblog" / logic_file / "cm"
    cm_dir.mkdir(parents=True, exist_ok=True)
    out_path = f"{cm_dir}/{experiment_name}_cm.png"
    
    plot_confusion_matrix(mat, classes, out_path)

Saved confusion matrix plot to: ../reports/aitv2/santos/deepproblog/ait/cm/ait_scratch_behavioral_w10_1000b1000a_20260426_135204_cm.png
Saved confusion matrix plot to: ../reports/aitv2/santos/deepproblog/ait/cm/ait_scratch_reduced_w10_1000b1000a_20260426_142318_cm.png
Saved confusion matrix plot to: ../reports/aitv2/santos/deepproblog/ait/cm/ait_scratch_reduced_w10_10000b10000a_20260426_140247_cm.png
Saved confusion matrix plot to: ../reports/aitv2/santos/deepproblog/ait_logic_baseline/cm/ait_logic_baseline_scratch_behavioral_w10_50b50a_20260428_184713_cm.png
Saved confusion matrix plot to: ../reports/aitv2/santos/deepproblog/ait_logic_baseline/cm/ait_logic_baseline_scratch_reduced_w10_100b100a_20260426_143747_cm.png
Saved confusion matrix plot to: ../reports/aitv2/santos/deepproblog/ait_logic_baseline/cm/ait_logic_baseline_scratch_behavioral_w10_50b50a_20260428_184524_cm.png


In [75]:
for model in baseline_models:

    folder = Path(f"../experiments/{dataset}/{scenario}/baselines/{model}/metrics")
    file_paths = list(folder.iterdir())
    
    for file_path in file_paths:
        experiment_name = file_path.stem
        print(f"Processing {experiment_name}...")
        
        with open(file_path) as f:
            metrics = json.load(f)

        cm = metrics["Confusion Matrix (actual_pred)"]
        mat = np.array(cm).T 
    
        cm_dir = plots_dir / "baselines" / model / "cm"
        cm_dir.mkdir(parents=True, exist_ok=True)
        out_path = f"{cm_dir}/{experiment_name}_cm.png"
    
        experiment_name = experiment_name[:-8]  # Remove "run_id"
        plot_confusion_matrix(mat, classes, experiment_name, out_path)
    

FileNotFoundError: [Errno 2] No such file or directory: '../experiments/aitv2/santos/baselines/multiclass/metrics'